# 📚 Smart Sinhala Buddhist Corpus Builder
# with Intelligent Text Extraction

## 🎯 Features:
- ✅ **Smart Detection**: Checks if PDF has extractable Unicode text first
- 💰 **Cost-Efficient**: Only uses Vision API for image-based PDFs
- 💾 **RAM-Optimized**: Processes and cleans up images immediately
- 📊 **Detailed Tracking**: Separate statistics for text-based vs. image-based extraction
- 🔍 **Quality Control**: Maintains same cleaning and validation pipeline

---

## 🔧 1. Setup & Installation

In [1]:
# ========================================
# 📦 INSTALL REQUIRED PACKAGES
# ========================================
print("Installing required packages...\n")

# Core PDF processing libraries
!pip install -q pdf2image PyPDF2 pdfplumber

# Vision API and authentication
!pip install -q google-cloud-vision google-auth

# Cleaning libraries
!pip install -q stanza langdetect

# System dependencies for PDF processing
!apt-get install -y poppler-utils > /dev/null 2>&1

print("✅ All packages installed successfully!")

Installing required packages...

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.8/42.8 kB 639.3 kB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.9/67.9 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 22.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 41.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.1/529.1 kB 8.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 15.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 55.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 31.5 MB/s eta 0:00:00
✅ All packages installed successfully!


In [3]:
# ========================================
# 📂 MOUNT GOOGLE DRIVE
# ========================================

from google.colab import drive
drive.mount('/content/drive')

print("\n✅ Google Drive mounted successfully!")

Mounted at /content/drive

✅ Google Drive mounted successfully!


In [4]:
VISION_API_KEY_PATH = '/content/drive/Othercomputers/My Mac/Multi-lingual-Buddhist-Conversational-Agent/auth/Vision OCR thematic runner.json'

In [5]:
import os
os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = VISION_API_KEY_PATH

## ⚙️ 2. Configuration & Settings

In [6]:
# ========================================
# 📁 FOLDER PATHS
# ========================================

# Input folder containing PDFs
PDF_INPUT_FOLDER = "/content/drive/Othercomputers/My Mac/Multi-lingual-Buddhist-Conversational-Agent/data/raw/sinhala_corpus"

# Output folder structure
BASE_OUTPUT = "/content/drive/Othercomputers/My Mac/Multi-lingual-Buddhist-Conversational-Agent/data/processed"

OUTPUT_RAW_TEXT = os.path.join(BASE_OUTPUT, "1_raw_text")
OUTPUT_CLEANED_TEXT = os.path.join(BASE_OUTPUT, "2_cleaned_text")
OUTPUT_CORPUS = os.path.join(BASE_OUTPUT, "3_final_corpus")
OUTPUT_LOGS = os.path.join(BASE_OUTPUT, "4_logs")
OUTPUT_TEMP_IMAGES = os.path.join(BASE_OUTPUT, "temp_images")  # Temporary storage

# Create output directories
for folder in [OUTPUT_RAW_TEXT, OUTPUT_CLEANED_TEXT, OUTPUT_CORPUS, OUTPUT_LOGS, OUTPUT_TEMP_IMAGES]:
    os.makedirs(folder, exist_ok=True)

print("✅ Folder structure created:")
print(f"  📥 Input: {PDF_INPUT_FOLDER}")
print(f"  📤 Raw texts: {OUTPUT_RAW_TEXT}")
print(f"  📤 Cleaned texts: {OUTPUT_CLEANED_TEXT}")
print(f"  📤 Final corpus: {OUTPUT_CORPUS}")
print(f"  📤 Logs: {OUTPUT_LOGS}")
print(f"  🗂️  Temp images: {OUTPUT_TEMP_IMAGES}")

✅ Folder structure created:
  📥 Input: /content/drive/Othercomputers/My Mac/Multi-lingual-Buddhist-Conversational-Agent/data/raw/sinhala_corpus
  📤 Raw texts: /content/drive/Othercomputers/My Mac/Multi-lingual-Buddhist-Conversational-Agent/data/processed/1_raw_text
  📤 Cleaned texts: /content/drive/Othercomputers/My Mac/Multi-lingual-Buddhist-Conversational-Agent/data/processed/2_cleaned_text
  📤 Final corpus: /content/drive/Othercomputers/My Mac/Multi-lingual-Buddhist-Conversational-Agent/data/processed/3_final_corpus
  📤 Logs: /content/drive/Othercomputers/My Mac/Multi-lingual-Buddhist-Conversational-Agent/data/processed/4_logs
  🗂️  Temp images: /content/drive/Othercomputers/My Mac/Multi-lingual-Buddhist-Conversational-Agent/data/processed/temp_images


In [7]:
# ========================================
# ⚙️ PROCESSING PARAMETERS
# ========================================

# Text extraction parameters
MIN_UNICODE_TEXT_LENGTH = 500  # Min characters to consider PDF as text-based
MIN_SINHALA_RATIO = 0.3  # Min % of Sinhala characters to validate text extraction

# Vision OCR parameters (only used for image-based PDFs)
OCR_DPI = 300  # Resolution for converting PDF pages to images

# Text cleaning parameters
MIN_PAGE_LENGTH = 50  # Minimum characters per page to keep
MAX_NON_SINHALA_RATIO = 0.7  # Maximum ratio of non-Sinhala text allowed
REMOVE_DUPLICATE_PAGES = True  # Remove duplicate pages

# Cost tracking (Vision API pricing as of 2024)
VISION_API_COST_PER_1000 = 1.50  # USD per 1000 images

print("⚙️  Configuration loaded:")
print(f"  📏 Min Unicode text length: {MIN_UNICODE_TEXT_LENGTH} chars")
print(f"  🇱🇰 Min Sinhala ratio: {MIN_SINHALA_RATIO:.0%}")
print(f"  🖼️  OCR DPI: {OCR_DPI}")
print(f"  💰 Vision API cost: ${VISION_API_COST_PER_1000} per 1000 images")

⚙️  Configuration loaded:
  📏 Min Unicode text length: 500 chars
  🇱🇰 Min Sinhala ratio: 30%
  🖼️  OCR DPI: 300
  💰 Vision API cost: $1.5 per 1000 images


## 🔍 3. Helper Functions

In [8]:
# ========================================
# 🔤 TEXT ANALYSIS FUNCTIONS
# ========================================

import re
from collections import Counter

def is_sinhala_char(char):
    """Check if a character is Sinhala."""
    return '\u0D80' <= char <= '\u0DFF'

def calculate_sinhala_ratio(text):
    """Calculate the ratio of Sinhala characters in text."""
    if not text:
        return 0.0
    sinhala_count = sum(1 for char in text if is_sinhala_char(char))
    return sinhala_count / len(text)

def is_valid_sinhala_text(text, min_length=MIN_UNICODE_TEXT_LENGTH, min_ratio=MIN_SINHALA_RATIO):
    """Check if text is valid Sinhala content."""
    if len(text.strip()) < min_length:
        return False
    sinhala_ratio = calculate_sinhala_ratio(text)
    return sinhala_ratio >= min_ratio

def clean_text(text):
    """Clean extracted text while preserving Sinhala content."""
    # Remove excessive whitespace
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'\n\s*\n', '\n\n', text)

    # Remove very short lines (likely artifacts)
    lines = text.split('\n')
    lines = [line.strip() for line in lines if len(line.strip()) >= 3]

    return '\n'.join(lines).strip()

print("✅ Text analysis functions loaded")

✅ Text analysis functions loaded


In [9]:
# ========================================
# 📄 PDF TEXT EXTRACTION
# ========================================

import PyPDF2
import pdfplumber

def extract_text_from_pdf_unicode(pdf_path):
    """
    Try to extract Unicode text from PDF using multiple methods.
    Returns (text, success, method_used)
    """

    # Method 1: Try pdfplumber (better for complex layouts)
    try:
        text_parts = []
        with pdfplumber.open(pdf_path) as pdf:
            for page in pdf.pages:
                page_text = page.extract_text()
                if page_text:
                    text_parts.append(page_text)

        text = '\n'.join(text_parts)
        if is_valid_sinhala_text(text):
            return text, True, "pdfplumber"
    except Exception as e:
        pass

    # Method 2: Try PyPDF2 (faster, simpler)
    try:
        text_parts = []
        with open(pdf_path, 'rb') as file:
            pdf_reader = PyPDF2.PdfReader(file)
            for page in pdf_reader.pages:
                page_text = page.extract_text()
                if page_text:
                    text_parts.append(page_text)

        text = '\n'.join(text_parts)
        if is_valid_sinhala_text(text):
            return text, True, "PyPDF2"
    except Exception as e:
        pass

    # No valid text found
    return "", False, "none"

print("✅ Unicode text extraction functions loaded")

✅ Unicode text extraction functions loaded


In [10]:
# ========================================
# 🖼️ VISION OCR FUNCTIONS
# ========================================

from pdf2image import convert_from_path
from google.cloud import vision
import io
import shutil
import psutil

def get_ram_usage_mb():
    """Get current RAM usage in MB."""
    process = psutil.Process()
    return process.memory_info().rss / 1024 / 1024

def extract_text_with_vision_ocr(pdf_path, book_name, max_pages=50):
    """
    Extract text from PDF using Vision API.
    Returns (text, num_pages_processed, ram_used_mb)
    """
    client = vision.ImageAnnotatorClient()

    # Create temporary folder for this book's images
    book_temp_folder = os.path.join(OUTPUT_TEMP_IMAGES, book_name.replace('.pdf', ''))
    os.makedirs(book_temp_folder, exist_ok=True)

    initial_ram = get_ram_usage_mb()

    try:
        # Convert PDF to images
        images = convert_from_path(pdf_path, dpi=OCR_DPI, output_folder=book_temp_folder, last_page=max_pages)
        num_pages = len(images)

        extracted_pages = []

        # Process each page
        for idx, image in enumerate(images, 1):
            # Save image temporarily
            img_path = os.path.join(book_temp_folder, f'page_{idx:04d}.jpg')
            image.save(img_path, 'JPEG')

            # Perform OCR
            with io.open(img_path, 'rb') as image_file:
                content = image_file.read()

            image_obj = vision.Image(content=content)
            response = client.document_text_detection(image=image_obj)

            if response.full_text_annotation:
                page_text = response.full_text_annotation.text
                extracted_pages.append(page_text)

            # Delete image immediately after processing to save RAM
            os.remove(img_path)

        # Combine all pages
        full_text = '\n\n'.join(extracted_pages)

        # Calculate RAM used
        final_ram = get_ram_usage_mb()
        ram_used = final_ram - initial_ram

        return full_text, num_pages, max(0, ram_used)

    finally:
        # Clean up temporary folder
        if os.path.exists(book_temp_folder):
            shutil.rmtree(book_temp_folder)

print("✅ Vision OCR functions loaded")

✅ Vision OCR functions loaded


## 🚀 4. Smart Text Extraction Pipeline

In [11]:
# ========================================
# 📚 SMART PDF PROCESSING
# ========================================

import json
from datetime import datetime
from tqdm.auto import tqdm

# Initialize tracking variables
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

processing_log = []
successful_books = []
all_texts = []

# Statistics tracking
stats = {
    'total_books': 0,
    'unicode_extracted': 0,
    'vision_ocr_used': 0,
    'failed': 0,
    'total_vision_api_calls': 0,
    'total_ram_used_mb': 0,
    'estimated_cost_usd': 0
}

# Get list of PDFs
pdf_files = [f for f in os.listdir(PDF_INPUT_FOLDER) if f.lower().endswith('.pdf')]
stats['total_books'] = len(pdf_files)

print(f"\n{'='*70}")
print(f"🚀 STARTING SMART PDF PROCESSING")
print(f"{'='*70}")
print(f"📚 Total PDFs found: {len(pdf_files)}")
print(f"⏰ Start time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"\n💡 Strategy: Try Unicode extraction first → Use Vision OCR only if needed")
print(f"{'='*70}\n")

# Process each PDF
for pdf_file in tqdm(pdf_files, desc="📖 Processing PDFs"):
    pdf_path = os.path.join(PDF_INPUT_FOLDER, pdf_file)
    book_name = os.path.splitext(pdf_file)[0]

    log_entry = {
        'book_name': pdf_file,
        'timestamp': datetime.now().isoformat(),
        'extraction_method': None,
        'status': None,
        'text_length': 0,
        'sinhala_ratio': 0,
        'pages_processed': 0,
        'ram_used_mb': 0,
        'vision_api_cost_usd': 0,
        'error': None
    }

    try:
        print(f"\n📖 Processing: {pdf_file}")

        # STEP 1: Try Unicode text extraction
        print("  🔍 Attempting Unicode text extraction...")
        text, success, method = extract_text_from_pdf_unicode(pdf_path)

        if success:
            # Unicode extraction successful!
            print(f"  ✅ Unicode extraction successful using {method}")
            log_entry['extraction_method'] = f'unicode_{method}'
            stats['unicode_extracted'] += 1

        else:
            # STEP 2: Fall back to Vision OCR
            print("  ⚠️  No valid Unicode text found")
            print("  🖼️  Using Vision OCR API...")

            text, num_pages, ram_used = extract_text_with_vision_ocr(pdf_path, book_name, max_pages=100)

            # Track Vision API usage
            log_entry['extraction_method'] = 'vision_ocr'
            log_entry['pages_processed'] = num_pages
            log_entry['ram_used_mb'] = round(ram_used, 2)
            log_entry['vision_api_cost_usd'] = round((num_pages / 1000) * VISION_API_COST_PER_1000, 4)

            stats['vision_ocr_used'] += 1
            stats['total_vision_api_calls'] += num_pages
            stats['total_ram_used_mb'] += ram_used
            stats['estimated_cost_usd'] += log_entry['vision_api_cost_usd']

            print(f"  ✅ Vision OCR complete: {num_pages} pages, ~${log_entry['vision_api_cost_usd']:.4f}")

        # STEP 3: Clean and validate text
        text = clean_text(text)

        if len(text) < MIN_PAGE_LENGTH:
            raise ValueError("Text too short after cleaning")

        sinhala_ratio = calculate_sinhala_ratio(text)
        if sinhala_ratio < MIN_SINHALA_RATIO:
            raise ValueError(f"Sinhala ratio too low: {sinhala_ratio:.2%}")

        # STEP 4: Save extracted text
        raw_text_path = os.path.join(OUTPUT_RAW_TEXT, f"{book_name}_raw.txt")
        with open(raw_text_path, 'w', encoding='utf-8') as f:
            f.write(text)

        cleaned_text_path = os.path.join(OUTPUT_CLEANED_TEXT, f"{book_name}_cleaned.txt")
        with open(cleaned_text_path, 'w', encoding='utf-8') as f:
            f.write(text)

        # Update tracking
        log_entry['status'] = 'success'
        log_entry['text_length'] = len(text)
        log_entry['sinhala_ratio'] = round(sinhala_ratio, 3)

        successful_books.append(pdf_file)
        all_texts.append(text)

        print(f"  ✅ Success: {len(text):,} chars, {sinhala_ratio:.1%} Sinhala")

    except Exception as e:
        log_entry['status'] = 'failed'
        log_entry['error'] = str(e)
        stats['failed'] += 1
        print(f"  ❌ Failed: {str(e)}")

    processing_log.append(log_entry)

print(f"\n{'='*70}")
print(f"✅ PROCESSING COMPLETE!")
print(f"{'='*70}\n")


🚀 STARTING SMART PDF PROCESSING
📚 Total PDFs found: 50
⏰ Start time: 2025-11-02 14:07:21

💡 Strategy: Try Unicode extraction first → Use Vision OCR only if needed



📖 Processing PDFs:   0%|          | 0/50 [00:00<?, ?it/s]


📖 Processing: Copy of මහා නිද්දේස පාළි.pdf
  🔍 Attempting Unicode text extraction...
  ⚠️  No valid Unicode text found
  🖼️  Using Vision OCR API...


KeyboardInterrupt: 

## 📊 5. Results & Statistics

In [ ]:
# ========================================
# 📊 DISPLAY DETAILED STATISTICS
# ========================================

print(f"\n{'='*70}")
print(f"📊 PROCESSING STATISTICS")
print(f"{'='*70}\n")

print("📚 Overall Results:")
print(f"  Total books processed: {stats['total_books']}")
print(f"  ✅ Successful: {len(successful_books)}")
print(f"  ❌ Failed: {stats['failed']}")
print(f"  Success rate: {(len(successful_books)/stats['total_books']*100):.1f}%")

print(f"\n🔤 Extraction Methods:")
print(f"  📄 Unicode extraction: {stats['unicode_extracted']} books")
print(f"  🖼️  Vision OCR used: {stats['vision_ocr_used']} books")

if stats['vision_ocr_used'] > 0:
    print(f"\n💰 Vision API Usage & Costs:")
    print(f"  Total API calls: {stats['total_vision_api_calls']:,} pages")
    print(f"  Estimated cost: ${stats['estimated_cost_usd']:.4f} USD")
    print(f"  RAM used: {stats['total_ram_used_mb']:.1f} MB")
    print(f"  Avg pages per book: {stats['total_vision_api_calls']/stats['vision_ocr_used']:.1f}")
    print(f"  Avg cost per book: ${stats['estimated_cost_usd']/stats['vision_ocr_used']:.4f}")
else:
    print(f"\n💰 Vision API Usage:")
    print(f"  🎉 No Vision API calls needed! All PDFs had extractable Unicode text.")
    print(f"  💵 Total cost: $0.00 USD")

if len(all_texts) > 0:
    total_chars = sum(len(text) for text in all_texts)
    total_words = sum(len(text.split()) for text in all_texts)
    avg_sinhala = sum(calculate_sinhala_ratio(text) for text in all_texts) / len(all_texts)

    print(f"\n📝 Corpus Statistics:")
    print(f"  Total characters: {total_chars:,}")
    print(f"  Total words: {total_words:,}")
    print(f"  Average Sinhala ratio: {avg_sinhala:.1%}")
    print(f"  Avg chars per book: {total_chars/len(all_texts):,.0f}")

print(f"\n{'='*70}\n")

In [ ]:
# ========================================
# 💾 SAVE PROCESSING LOG
# ========================================

import csv

# Save as CSV
log_path = os.path.join(OUTPUT_LOGS, f'processing_log_{timestamp}.csv')
with open(log_path, 'w', newline='', encoding='utf-8') as f:
    if processing_log:
        writer = csv.DictWriter(f, fieldnames=processing_log[0].keys())
        writer.writeheader()
        writer.writerows(processing_log)

print(f"✅ Processing log saved to: {log_path}")

# Save detailed summary
summary_path = os.path.join(OUTPUT_LOGS, f'summary_{timestamp}.json')
summary = {
    'timestamp': timestamp,
    'statistics': stats,
    'successful_books': successful_books,
    'failed_count': stats['failed']
}

with open(summary_path, 'w', encoding='utf-8') as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)

print(f"✅ Summary saved to: {summary_path}")

## 📦 6. Create Final Corpus

In [ ]:
# ========================================
# 📄 CREATE COMBINED CORPUS
# ========================================

if all_texts:
    # Combine all texts
    combined_corpus = '\n\n'.join(cleaned_corpus)

    # Save combined corpus
    corpus_filename = f'buddhist_corpus_combined_{timestamp}.txt'
    corpus_path = os.path.join(OUTPUT_CORPUS, corpus_filename)

    with open(corpus_path, 'w', encoding='utf-8') as f:
        f.write(combined_corpus)

    print(f"✅ Combined corpus saved to: {corpus_path}")
    print(f"   Size: {len(combined_corpus):,} characters")

    # Create JSONL format
    jsonl_filename = f'buddhist_corpus_{timestamp}.jsonl'
    jsonl_path = os.path.join(OUTPUT_CORPUS, jsonl_filename)

    with open(jsonl_path, 'w', encoding='utf-8') as f:
        for book_name, text in zip(successful_books, all_texts):
            entry = {
                'text': text,
                'source': book_name,
                'language': 'si',
                'length': len(text),
                'sinhala_ratio': calculate_sinhala_ratio(text)
            }
            f.write(json.dumps(entry, ensure_ascii=False) + '\n')

    print(f"✅ JSONL corpus saved to: {jsonl_path}")
else:
    print("⚠️  No texts to combine!")

In [ ]:
# ========================================
# 📋 CREATE METADATA FILE
# ========================================

metadata = {
    'corpus_name': 'Sinhala Buddhist Texts Corpus (Smart Extraction)',
    'creation_date': timestamp,
    'books_included': successful_books,
    'total_books': len(successful_books),
    'extraction_statistics': {
        'unicode_extracted': stats['unicode_extracted'],
        'vision_ocr_used': stats['vision_ocr_used'],
        'failed': stats['failed']
    },
    'vision_api_usage': {
        'total_api_calls': stats['total_vision_api_calls'],
        'estimated_cost_usd': round(stats['estimated_cost_usd'], 4),
        'ram_used_mb': round(stats['total_ram_used_mb'], 2)
    },
    'corpus_metrics': {
        'total_characters': len(combined_corpus) if all_texts else 0,
        'total_words': len(combined_corpus.split()) if all_texts else 0,
        'average_sinhala_ratio': round(sum(calculate_sinhala_ratio(text) for text in all_texts) / len(all_texts), 3) if all_texts else 0
    },
    'processing_parameters': {
        'min_unicode_text_length': MIN_UNICODE_TEXT_LENGTH,
        'min_sinhala_ratio': MIN_SINHALA_RATIO,
        'ocr_dpi': OCR_DPI,
        'min_page_length': MIN_PAGE_LENGTH,
        'max_non_sinhala_ratio': MAX_NON_SINHALA_RATIO,
        'remove_duplicates': REMOVE_DUPLICATE_PAGES
    },
    'file_formats': {
        'combined_text': corpus_filename if all_texts else None,
        'jsonl': jsonl_filename if all_texts else None,
        'individual_cleaned_texts': OUTPUT_CLEANED_TEXT
    }
}

metadata_path = os.path.join(OUTPUT_CORPUS, f'corpus_metadata_{timestamp}.json')
with open(metadata_path, 'w', encoding='utf-8') as f:
    json.dump(metadata, f, indent=2, ensure_ascii=False)

print(f"\n✅ Metadata saved to: {metadata_path}")

In [ ]:
# ========================================
# 📁 MOVE PROCESSED PDFs
# ========================================

import shutil

processed_folder = "/content/drive/Othercomputers/My Mac/Multi-lingual-Buddhist-Conversational-Agent/data/processed/5_pdfs/"
os.makedirs(processed_folder, exist_ok=True)

print(f"\n📦 Moving processed PDFs to: {processed_folder}")

for book in successful_books:
    source = os.path.join(PDF_INPUT_FOLDER, book)
    destination = os.path.join(processed_folder, book)
    try:
        shutil.move(source, destination)
        print(f"  ✅ Moved: {book}")
    except Exception as e:
        print(f"  ⚠️  Could not move {book}: {e}")

print("\n✅ File organization complete!")

## 🎯 7. Final Summary & Cost Report

In [ ]:
# ========================================
# 🎉 FINAL SUMMARY
# ========================================

print("\n" + "="*80)
print("🎉 CORPUS BUILDING COMPLETE!")
print("="*80)

print(f"\n📊 Processing Summary:")
print(f"  Total PDFs: {stats['total_books']}")
print(f"  ✅ Successfully processed: {len(successful_books)} books")
print(f"  ❌ Failed: {stats['failed']} books")
print(f"  Success rate: {(len(successful_books)/stats['total_books']*100):.1f}%")

print(f"\n🔤 Extraction Breakdown:")
print(f"  📄 Unicode extraction: {stats['unicode_extracted']} books ({stats['unicode_extracted']/len(successful_books)*100:.1f}% of successful)")
print(f"  🖼️  Vision OCR required: {stats['vision_ocr_used']} books ({stats['vision_ocr_used']/len(successful_books)*100:.1f}% of successful)")

if stats['vision_ocr_used'] > 0:
    print(f"\n💰 Vision API Cost Analysis:")
    print(f"  Total pages processed: {stats['total_vision_api_calls']:,}")
    print(f"  💵 Estimated cost: ${stats['estimated_cost_usd']:.4f} USD")
    print(f"  📊 Cost per book: ${stats['estimated_cost_usd']/stats['vision_ocr_used']:.4f}")
    print(f"  📊 Cost per page: ${stats['estimated_cost_usd']/stats['total_vision_api_calls']:.5f}")
    print(f"  💾 Total RAM used: {stats['total_ram_used_mb']:.1f} MB")

    # Calculate savings
    potential_cost = (stats['total_books'] * stats['total_vision_api_calls'] / stats['vision_ocr_used']) / 1000 * VISION_API_COST_PER_1000
    savings = potential_cost - stats['estimated_cost_usd']
    print(f"\n💡 Smart Extraction Savings:")
    print(f"  If all PDFs used Vision API: ${potential_cost:.4f}")
    print(f"  Actual cost: ${stats['estimated_cost_usd']:.4f}")
    print(f"  💰 Money saved: ${savings:.4f} ({savings/potential_cost*100:.1f}%)")
else:
    print(f"\n💰 Vision API Cost:")
    print(f"  🎉 $0.00 - All PDFs had Unicode text!")

if all_texts:
    print(f"\n📝 Final Corpus Statistics:")
    print(f"  Total characters: {len(combined_corpus):,}")
    print(f"  Total words: {len(combined_corpus.split()):,}")
    print(f"  Average Sinhala ratio: {sum(calculate_sinhala_ratio(text) for text in all_texts) / len(all_texts):.1%}")

print(f"\n📁 Output Files:")
print(f"  📄 Combined corpus: {corpus_path if all_texts else 'N/A'}")
print(f"  📦 JSONL format: {jsonl_path if all_texts else 'N/A'}")
print(f"  📋 Processing log: {log_path}")
print(f"  📊 Summary report: {summary_path}")
print(f"  🔍 Metadata: {metadata_path}")

print(f"\n📂 Additional Folders:")
print(f"  Raw texts: {OUTPUT_RAW_TEXT}")
print(f"  Cleaned texts: {OUTPUT_CLEANED_TEXT}")
print(f"  Processed PDFs moved to: {processed_folder}")

print("\n" + "="*80)
print("💡 Next Steps:")
print("  1. Review processing log for extraction methods used")
print("  2. Check Vision API usage and costs")
print("  3. Verify text quality for both extraction methods")
print("  4. Use corpus for model training/fine-tuning")
print("="*80)

print(f"\n⏰ Processing completed at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("\n✨ Thank you for using the Smart Buddhist Corpus Builder! ✨\n")

## 🔍 8. Quality Check & Inspection

In [ ]:
# ========================================
# 🔍 COMPARE EXTRACTION METHODS
# ========================================

# Show sample from each extraction method
unicode_books = [log for log in processing_log if log['extraction_method'] and 'unicode' in log['extraction_method'] and log['status'] == 'success']
vision_books = [log for log in processing_log if log['extraction_method'] == 'vision_ocr' and log['status'] == 'success']

if unicode_books:
    sample = unicode_books[0]
    book_idx = successful_books.index(sample['book_name'])
    text = all_texts[book_idx]

    print("📄 Sample from Unicode Extraction:")
    print("="*60)
    print(f"Book: {sample['book_name']}")
    print(f"Method: {sample['extraction_method']}")
    print(f"Length: {len(text):,} chars")
    print(f"Sinhala ratio: {calculate_sinhala_ratio(text):.1%}")
    print("\nFirst 500 characters:")
    print("-"*60)
    print(text[:500])
    print("-"*60)

if vision_books:
    sample = vision_books[0]
    book_idx = successful_books.index(sample['book_name'])
    text = all_texts[book_idx]

    print("\n🖼️  Sample from Vision OCR:")
    print("="*60)
    print(f"Book: {sample['book_name']}")
    print(f"Pages processed: {sample['pages_processed']}")
    print(f"Cost: ${sample['vision_api_cost_usd']:.4f}")
    print(f"RAM used: {sample['ram_used_mb']:.2f} MB")
    print(f"Length: {len(text):,} chars")
    print(f"Sinhala ratio: {calculate_sinhala_ratio(text):.1%}")
    print("\nFirst 500 characters:")
    print("-"*60)
    print(text[:500])
    print("-"*60)

In [ ]:
# ========================================
# 📊 EXTRACTION METHOD STATISTICS
# ========================================

if successful_books:
    # Group by extraction method
    methods = {}
    for log in processing_log:
        if log['status'] == 'success':
            method = log['extraction_method']
            if method not in methods:
                methods[method] = []
            methods[method].append(log)

    print("\n📊 Extraction Method Analysis:")
    print("="*70)

    for method, logs in methods.items():
        avg_length = sum(log['text_length'] for log in logs) / len(logs)
        avg_sinhala = sum(log['sinhala_ratio'] for log in logs) / len(logs)

        print(f"\n{method.upper()}:")
        print(f"  Books: {len(logs)}")
        print(f"  Avg text length: {avg_length:,.0f} characters")
        print(f"  Avg Sinhala ratio: {avg_sinhala:.1%}")

        if 'vision_ocr' in method:
            total_pages = sum(log['pages_processed'] for log in logs)
            total_cost = sum(log['vision_api_cost_usd'] for log in logs)
            total_ram = sum(log['ram_used_mb'] for log in logs)

            print(f"  Total pages: {total_pages}")
            print(f"  Total cost: ${total_cost:.4f}")
            print(f"  Total RAM: {total_ram:.1f} MB")

    print("\n" + "="*70)